In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Graph Neural Networks.

Contains: Cora dataset loading (with Karate Club fallback), graph
normalisation, train/val/test split, kailash-ml engine setup,
graph visualisation helpers, and training harness.

Technique-specific code (GCN, GAT, GraphSAGE layers) does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex6_gnns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cora"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# GRAPH DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_cora() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Cora — 2708 papers, 1433 bag-of-words features, 7 classes.

    Returns:
        X_np: node features (N, F)
        A_np: dense adjacency matrix (N, N)
        y_np: node labels (N,)
        edge_index_np: edge list (2, E) for link prediction
        dataset_name: "Cora"
        n_classes: 7
    """
    from torch_geometric.datasets import Planetoid

    dataset = Planetoid(root=str(DATA_DIR), name="Cora")
    # torch_geometric.data.Data has dynamic attributes (num_nodes/x/y/edge_index);
    # use Any to avoid type-checker false positives without losing runtime fidelity.
    data: Any = dataset[0]
    n = data.num_nodes
    X_np = data.x.numpy().astype(np.float32)
    y_np = data.y.numpy().astype(np.int64)

    # Build a dense adjacency matrix from the edge_index. Cora has ~10k
    # directed edges (5278 undirected) over 2708 nodes; the dense matrix
    # is ~7M entries which fits comfortably in CPU memory.
    A_np = np.zeros((n, n), dtype=np.float32)
    edge_index_np = data.edge_index.numpy()
    src = edge_index_np[0]
    dst = edge_index_np[1]
    A_np[src, dst] = 1.0
    A_np[dst, src] = 1.0  # symmetrise just in case
    n_classes = int(dataset.num_classes)
    return X_np, A_np, y_np, edge_index_np, "Cora", n_classes


def load_karate() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Zachary's Karate Club — 34 nodes, 78 edges, 2 factions."""
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A_np = nx.to_numpy_array(G, dtype=np.float32)
    labels = np.array(
        [0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)],
        dtype=np.int64,
    )
    # Karate has no node features; use one-hot identity (transductive)
    X_np = np.eye(n, dtype=np.float32)
    # Build edge_index from adjacency
    src, dst = np.where(A_np > 0)
    edge_index_np = np.stack([src, dst]).astype(np.int64)
    return X_np, A_np, labels, edge_index_np, "Karate Club", 2


def load_graph_data() -> dict:
    """Load Cora (with Karate fallback) and return all graph tensors.

    Returns a dict with keys:
        X, A, y, A_norm, A_hat — torch tensors on device
        X_np, A_np, y_np, edge_index_np — numpy arrays
        train_mask, val_mask, test_mask — boolean masks on device
        N, F_dim, n_classes, n_edges_undirected, dataset_name — scalars
    """
    try:
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_cora()
    except Exception as exc:
        print(
            f"Could not load Cora ({type(exc).__name__}: {exc}); "
            "falling back to Karate Club"
        )
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_karate()

    N = X_np.shape[0]
    F_dim = X_np.shape[1]
    n_edges_undirected = int(A_np.sum() // 2)
    print(
        f"Graph: {dataset_name} — {N} nodes, {n_edges_undirected} undirected edges, "
        f"feature_dim={F_dim}, classes={n_classes}"
    )
    class_counts = ", ".join(f"c{c}={int((y_np == c).sum())}" for c in range(n_classes))
    print(f"  per-class counts: {class_counts}")

    X = torch.from_numpy(X_np).to(device)
    A = torch.from_numpy(A_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Add self-loops and build the symmetric Laplacian D^{-1/2} A D^{-1/2}
    A_hat = A + torch.eye(N, device=device)
    deg = A_hat.sum(dim=1)
    d_inv_sqrt = deg.pow(-0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    A_norm = d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)

    # Train/val/test split — 20% train, 20% val, 60% test (per class)
    train_mask = torch.zeros(N, dtype=torch.bool, device=device)
    val_mask = torch.zeros(N, dtype=torch.bool, device=device)
    rng = np.random.default_rng(0)
    for c in range(n_classes):
        idx = np.where(y_np == c)[0]
        if len(idx) == 0:
            continue
        rng.shuffle(idx)
        n_train = max(1, int(0.2 * len(idx)))
        n_val = max(1, int(0.2 * len(idx)))
        train_mask[idx[:n_train]] = True
        val_mask[idx[n_train : n_train + n_val]] = True
    test_mask = ~(train_mask | val_mask)
    print(
        f"  train: {int(train_mask.sum().item())}, "
        f"val: {int(val_mask.sum().item())}, "
        f"test: {int(test_mask.sum().item())}"
    )

    return {
        "X": X,
        "A": A,
        "y": y,
        "A_norm": A_norm,
        "A_hat": A_hat,
        "X_np": X_np,
        "A_np": A_np,
        "y_np": y_np,
        "edge_index_np": edge_index_np,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "N": N,
        "F_dim": F_dim,
        "n_classes": n_classes,
        "n_edges_undirected": n_edges_undirected,
        "dataset_name": dataset_name,
    }


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_gnns.db"
    registry_db = "sqlite:///mlfp05_gnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_gnns", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS
# ════════════════════════════════════════════════════════════════════════


def train_node_classifier(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Train a GNN for node classification and log metrics to ExperimentTracker.

    Returns:
        train_losses: per-epoch training loss
        val_accs: per-epoch validation accuracy
        test_accs: per-epoch test accuracy
    """
    return asyncio.run(
        _train_node_classifier_async(
            model,
            name,
            forward_arg,
            graph_data,
            tracker,
            exp_name,
            epochs,
            lr,
            weight_decay,
        )
    )


async def _train_node_classifier_async(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Async core — uses the kailash-ml 1.1.1 ``tracker.track(...)`` context manager."""
    X = graph_data["X"]
    y = graph_data["y"]
    train_mask = graph_data["train_mask"]
    val_mask = graph_data["val_mask"]
    test_mask = graph_data["test_mask"]
    N = graph_data["N"]
    n_edges = graph_data["n_edges_undirected"]
    dataset_name = graph_data["dataset_name"]
    hidden_dim = 16 if dataset_name == "Karate Club" else 64

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] parameters: {n_params:,}")

    train_losses: list[float] = []
    val_accs: list[float] = []
    test_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(hidden_dim),
                "epochs": str(epochs),
                "lr": str(lr),
                "weight_decay": str(weight_decay),
                "n_params": str(n_params),
                "dataset": dataset_name,
                "n_nodes": str(N),
                "n_edges": str(n_edges),
            }
        )

        for epoch in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(X, forward_arg)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                preds = model(X, forward_arg).argmax(dim=-1)
                v_acc = (preds[val_mask] == y[val_mask]).float().mean().item()
                t_acc = (preds[test_mask] == y[test_mask]).float().mean().item()
            val_accs.append(v_acc)
            test_accs.append(t_acc)

            await run.log_metrics(
                {
                    "train_loss": loss.item(),
                    "val_accuracy": v_acc,
                    "test_accuracy": t_acc,
                },
                step=epoch + 1,
            )

            if (epoch + 1) % 25 == 0:
                print(
                    f"  [{name}] epoch {epoch+1:3d}  "
                    f"loss={loss.item():.4f}  val_acc={v_acc:.3f}  test_acc={t_acc:.3f}"
                )

        await run.log_metrics(
            {
                "final_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
                "final_test_accuracy": test_accs[-1],
                "best_val_accuracy": max(val_accs),
                "best_test_accuracy": max(test_accs),
            }
        )

    return train_losses, val_accs, test_accs


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════

viz = ModelVisualizer()


def plot_training_curves(
    metrics_dict: dict[str, list[float]],
    title: str,
    y_label: str,
    filename: str,
) -> None:
    """Plot overlaid training curves and save as HTML."""
    fig = viz.training_history(
        metrics=metrics_dict,
        x_label="Epoch",
        y_label=y_label,
    )
    fig.update_layout(title=title)
    filepath = OUTPUT_DIR / filename
    fig.write_html(str(filepath))
    print(f"  Saved: {filepath}")


def plot_node_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
) -> None:
    """2-D PCA projection of node embeddings coloured by class label.

    Uses SVD-based PCA (no sklearn dependency). Nodes of the same class
    should cluster together if the GNN learned meaningful representations.
    """
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    Vt = np.linalg.svd(emb_centered, full_matrices=False)[2]
    coords = emb_centered @ Vt.T[:, :2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap = plt.cm.get_cmap("tab10", n_classes)
    for c in range(n_classes):
        mask = labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=15,
            alpha=0.6,
            label=f"Class {c}",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    # Text summary of first 3 nodes per class
    print(f"\n  {title} — first 3 nodes per class:")
    for c in range(min(n_classes, 7)):
        rows = coords[labels == c][:3]
        if len(rows) == 0:
            continue
        pretty = ", ".join(f"({r[0]:+.2f}, {r[1]:+.2f})" for r in rows)
        print(f"    class {c}: {pretty}")

    return coords


def plot_graph_with_embeddings(
    A_np: np.ndarray,
    embeddings_2d: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
    max_nodes: int = 200,
) -> None:
    """Draw graph edges on the 2-D embedding space, coloured by class.

    Subsamples to max_nodes for readability on large graphs.
    """
    N = A_np.shape[0]
    if N > max_nodes:
        rng = np.random.default_rng(42)
        subset = rng.choice(N, max_nodes, replace=False)
    else:
        subset = np.arange(N)

    coords = embeddings_2d[subset]
    sub_labels = labels[subset]
    sub_A = A_np[np.ix_(subset, subset)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    cmap = plt.cm.get_cmap("tab10", n_classes)

    # Draw edges first (behind nodes)
    src_idx, dst_idx = np.where(np.triu(sub_A) > 0)
    for s, d in zip(src_idx, dst_idx):
        ax.plot(
            [coords[s, 0], coords[d, 0]],
            [coords[s, 1], coords[d, 1]],
            color="gray",
            alpha=0.08,
            linewidth=0.5,
        )

    # Draw nodes
    for c in range(n_classes):
        mask = sub_labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=25,
            alpha=0.7,
            label=f"Class {c}",
            zorder=2,
        )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


def plot_attention_weights(
    alpha: np.ndarray,
    A_np: np.ndarray,
    labels: np.ndarray,
    title: str,
    filename: str,
    node_idx: int = 0,
    top_k: int = 10,
) -> None:
    """Visualise attention weights for a single node's neighbourhood.

    Shows which neighbours the GAT layer attends to most strongly.
    """
    neighbours = np.where(A_np[node_idx] > 0)[0]
    if len(neighbours) == 0:
        print(f"  Node {node_idx} has no neighbours — skipping attention plot")
        return

    weights = alpha[node_idx, neighbours]
    order = np.argsort(-weights)[:top_k]
    top_neighbours = neighbours[order]
    top_weights = weights[order]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    bar_colors = [plt.cm.get_cmap("tab10")(labels[n] % 10) for n in top_neighbours]
    ax.barh(
        range(len(top_neighbours)),
        top_weights,
        color=bar_colors,
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_neighbours)))
    ax.set_yticklabels(
        [f"Node {n} (class {labels[n]})" for n in top_neighbours],
        fontsize=9,
    )
    ax.set_xlabel("Attention Weight")
    ax.set_title(
        f"{title}\nNode {node_idx} (class {labels[node_idx]}) attending to top-{top_k} neighbours",
        fontsize=12,
        fontweight="bold",
    )
    ax.invert_yaxis()
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Register a model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metrics,
    )
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Synchronously register a model."""
    return asyncio.run(_register_model(registry, name, model, metrics))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 6.5: GNN Architecture Comparison
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Systematic comparison: GCN vs GAT vs GraphSAGE on the same dataset
#   - Metrics that matter: accuracy, convergence speed, parameter count
#   - When to choose each architecture (decision framework)
#   - Register the best model in kailash-ml ModelRegistry
#   - Visualise side-by-side training curves and embedding quality
#
# PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_6.2 (GAT), M5/ex_6.3 (GraphSAGE).
# ESTIMATED TIME: ~25 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import pickle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt



## SETUP — Load Data and Build All Three Architectures


In [ ]:
print("=" * 70)
print("  GNN Architecture Comparison: GCN vs GAT vs GraphSAGE")
print("=" * 70)

graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
A_norm = graph_data["A_norm"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100
SAMPLE_K = 10



### GCN


In [ ]:
# TODO: Implement GCNLayer and GCN (same as exercise 6.1)
class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: self.W = nn.Linear(in_dim, out_dim, bias=True)
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: return a_norm @ self.W(h)
        pass


class GCN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: self.l1 = GCNLayer(in_dim, hidden_dim)
        #       self.l2 = GCNLayer(hidden_dim, n_classes)
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: ReLU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: return F.relu(self.l1(h, a_norm))
        pass



### GAT


In [ ]:
# TODO: Implement GATLayer and GAT (same as exercise 6.2)
class GATLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: self.W, self.a_src, self.a_dst — all nn.Linear, no bias
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Project, compute attention scores, mask, softmax, aggregate
        # Wh = self.W(h)
        # e_src = self.a_src(Wh); e_dst = self.a_dst(Wh)
        # scores = F.leaky_relu(e_src + e_dst.T, negative_slope=0.2)
        # mask = adj + torch.eye(adj.size(0), device=adj.device)
        # scores = scores.masked_fill(mask == 0, float("-inf"))
        # alpha = F.softmax(scores, dim=1)
        # return alpha @ Wh
        pass


class GAT(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: self.l1 = GATLayer(in_dim, hidden_dim)
        #       self.l2 = GATLayer(hidden_dim, n_classes)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: ELU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: return F.elu(self.l1(h, adj))
        pass



### GraphSAGE


In [ ]:
# TODO: Implement GraphSAGELayer and GraphSAGE (same as exercise 6.3)
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, sample_k: int = 10):
        super().__init__()
        # TODO: self.W_self, self.W_neigh — nn.Linear, no bias
        #       self.sample_k = sample_k
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        n = h.size(0)
        # TODO: Neighbour sampling (training only), mean aggregation, combine
        # if self.training and self.sample_k < n:
        #     sample_mask = torch.zeros_like(adj)
        #     for i in range(n):
        #         neigh_idx = torch.where(adj[i] > 0)[0]
        #         if len(neigh_idx) <= self.sample_k:
        #             sample_mask[i, neigh_idx] = 1.0
        #         else:
        #             perm = torch.randperm(len(neigh_idx), device=h.device)[:self.sample_k]
        #             sample_mask[i, neigh_idx[perm]] = 1.0
        #         adj_sampled = sample_mask
        # else:
        #     adj_sampled = adj
        # deg_sampled = adj_sampled.sum(dim=1, keepdim=True).clamp(min=1.0)
        # h_neigh = (adj_sampled @ h) / deg_sampled
        # return self.W_self(h) + self.W_neigh(h_neigh)
        pass


class GraphSAGE(nn.Module):
    def __init__(
        self, in_dim: int, hidden_dim: int, n_classes: int, sample_k: int = 10
    ):
        super().__init__()
        # TODO: self.l1 = GraphSAGELayer(in_dim, hidden_dim, sample_k)
        #       self.l2 = GraphSAGELayer(hidden_dim, n_classes, sample_k)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: ReLU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: return F.relu(self.l1(h, adj))
        pass



## TRAIN — All Three Architectures Under Identical Conditions


In [ ]:
print(f"\n== Training all three architectures on {dataset_name} ==\n")

# GCN
print(f"--- GCN ---")
gcn = GCN(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
gcn_losses, gcn_val, gcn_test = train_node_classifier(
    gcn,
    "GCN",
    A_norm,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)

# GAT
print(f"\n--- GAT ---")
gat = GAT(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
gat_losses, gat_val, gat_test = train_node_classifier(
    gat,
    "GAT",
    A,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)

# GraphSAGE
print(f"\n--- GraphSAGE ---")
sage = GraphSAGE(
    in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes, sample_k=SAMPLE_K
)
sage_losses, sage_val, sage_test = train_node_classifier(
    sage,
    "GraphSAGE",
    A,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)



## COMPARE — Quantitative Metrics Table


In [ ]:
print("\n" + "=" * 70)
print("  QUANTITATIVE COMPARISON")
print("=" * 70)

results = {
    "GCN": {
        "val_accs": gcn_val,
        "test_accs": gcn_test,
        "losses": gcn_losses,
        "model": gcn,
        "forward_arg": A_norm,
    },
    "GAT": {
        "val_accs": gat_val,
        "test_accs": gat_test,
        "losses": gat_losses,
        "model": gat,
        "forward_arg": A,
    },
    "GraphSAGE": {
        "val_accs": sage_val,
        "test_accs": sage_test,
        "losses": sage_losses,
        "model": sage,
        "forward_arg": A,
    },
}

# TODO: Print comparison table
# For each model, compute:
#   - n_params: sum(p.numel() for p in model.parameters())
#   - best_val: max(val_accs)
#   - best_test: max(test_accs)
#   - final_loss: losses[-1]
#   - conv_epoch: first epoch where val_acc >= 0.9 * best_val
# Hint: conv_epoch = next((i+1 for i, a in enumerate(val_accs) if a >= 0.9*best_val), EPOCHS)
print(
    f"\n{'Model':>12} {'Params':>8} {'Best Val':>10} {'Best Test':>10} "
    f"{'Final Loss':>12} {'Conv@90%':>10}"
)
print("-" * 66)

for name, r in results.items():
    n_params = sum(p.numel() for p in r["model"].parameters())
    best_val = max(r["val_accs"])
    best_test = max(r["test_accs"])
    final_loss = r["losses"][-1]
    # TODO: Compute convergence epoch (first epoch reaching 90% of best val accuracy)
    threshold = 0.9 * best_val
    conv_epoch = next(
        (i + 1 for i, a in enumerate(r["val_accs"]) if a >= threshold),
        EPOCHS,
    )
    print(
        f"{name:>12} {n_params:>8,} {best_val:>10.4f} {best_test:>10.4f} "
        f"{final_loss:>12.4f} {conv_epoch:>10}"
    )

# Determine the best model by best validation accuracy
best_name = max(results, key=lambda k: max(results[k]["val_accs"]))
best_model_obj = results[best_name]["model"]
best_val_acc = max(results[best_name]["val_accs"])
best_test_acc = max(results[best_name]["test_accs"])
print(f"\nBest model by validation accuracy: {best_name} ({best_val_acc:.4f})")



### Comparison Checkpoint


In [ ]:
assert len(results) == 3, "Should have results for all 3 architectures"
assert all(
    max(r["val_accs"]) > 0.3 for r in results.values()
), "All models should achieve > 30% accuracy (well above random for 7 classes)"
# INTERPRETATION: The comparison reveals architectural trade-offs:
# - GCN is simplest (fewest params) but uses fixed aggregation weights
# - GAT adds learnable attention but costs more parameters
# - GraphSAGE separates self vs neighbour projections and uses sampling
# Cora is a homogeneous citation graph where all three tend to perform
# similarly. Differences become more pronounced on heterogeneous graphs.
print("\n--- Comparison checkpoint passed --- all models evaluated\n")



## VISUALISE — Side-by-Side Training Curves and Embeddings


In [ ]:
print("=" * 70)
print("  VISUALISATIONS — Side-by-Side Comparison")
print("=" * 70)

# Plot 1: Loss curves overlay
plot_training_curves(
    metrics_dict={
        "GCN train loss": gcn_losses,
        "GAT train loss": gat_losses,
        "GraphSAGE train loss": sage_losses,
    },
    title="GNN Training Loss Comparison",
    y_label="Cross-Entropy Loss",
    filename="comparison_loss_curves.html",
)

# Plot 2: Accuracy curves overlay
plot_training_curves(
    metrics_dict={
        "GCN val acc": gcn_val,
        "GAT val acc": gat_val,
        "GraphSAGE val acc": sage_val,
    },
    title="GNN Validation Accuracy Comparison",
    y_label="Validation Accuracy",
    filename="comparison_accuracy_curves.html",
)

# TODO: Create side-by-side embedding visualisations (1 row, 3 columns)
# For each model:
# 1. Extract embeddings via model.embed()
# 2. PCA to 2D using SVD
# 3. Scatter plot coloured by class
# 4. Title with model name, val accuracy, and param count
# Hint: emb_centered = emb - emb.mean(axis=0, keepdims=True)
#        U, S, Vt = np.linalg.svd(emb_centered, full_matrices=False)
#        coords = emb_centered @ Vt.T[:, :2]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (name, r) in enumerate(results.items()):
    model = r["model"]
    model.eval()
    with torch.no_grad():
        if name == "GCN":
            emb = model.embed(X, A_norm).cpu().numpy()
        else:
            emb = model.embed(X, A).cpu().numpy()

    # TODO: PCA to 2D and scatter plot for each model
    # emb_centered = emb - emb.mean(axis=0, keepdims=True)
    # U, S, Vt = np.linalg.svd(emb_centered, full_matrices=False)
    # coords = emb_centered @ Vt.T[:, :2]
    # cmap = plt.cm.get_cmap("tab10", n_classes)
    # for c in range(n_classes):
    #     mask = y_np == c
    #     axes[idx].scatter(coords[mask, 0], coords[mask, 1], c=[cmap(c)], s=8, alpha=0.5, label=f"Class {c}")
    # n_params = sum(p.numel() for p in model.parameters())
    # best_v = max(r["val_accs"])
    # axes[idx].set_title(f"{name}\nval={best_v:.3f}, params={n_params:,}")
    pass

fig.suptitle(
    f"Node Embedding Comparison — {dataset_name}",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
filepath = OUTPUT_DIR / "comparison_embeddings.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# TODO: Create parameter efficiency plot (scatter: params vs accuracy)
# x-axis: number of parameters per model
# y-axis: best validation accuracy
# Annotate each point with the model name
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
model_names = list(results.keys())
param_counts = [
    sum(p.numel() for p in results[n]["model"].parameters()) for n in model_names
]
best_vals = [max(results[n]["val_accs"]) for n in model_names]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

# TODO: Scatter plot with annotations
# ax.scatter(param_counts, best_vals, s=200, c=colors, zorder=3, edgecolor="white", linewidth=2)
# for i, name in enumerate(model_names):
#     ax.annotate(name, (param_counts[i], best_vals[i]), textcoords="offset points", xytext=(10, 10))
# ax.set_xlabel("Number of Parameters")
# ax.set_ylabel("Best Validation Accuracy")
# ax.set_title("Parameter Efficiency: Accuracy vs Model Size")
# ax.grid(True, alpha=0.3)

plt.tight_layout()
filepath = OUTPUT_DIR / "comparison_param_efficiency.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

print("\n--- Visualisations saved ---\n")



## DECISION FRAMEWORK — When to Use Each Architecture


ARCHITECTURE SELECTION GUIDE:

  ┌─────────────────┬───────────────────────────────────────────────────┐
  │ Choose GCN when │ - Graph is small-to-medium (< 100K nodes)        │
  │                 │ - Neighbours are roughly equally important        │
  │                 │ - You want the simplest, fastest baseline         │
  │                 │ - Interpretability of attention is NOT needed     │
  │                 │ - Example: citation networks, molecular graphs    │
  ├─────────────────┼───────────────────────────────────────────────────┤
  │ Choose GAT when │ - Edge importance varies (some neighbours matter │
  │                 │   more than others)                               │
  │                 │ - You need INTERPRETABLE attention weights        │
  │                 │ - Regulated domain (finance, healthcare) needs    │
  │                 │   explainable model decisions                     │
  │                 │ - Example: fraud detection, drug interaction      │
  ├─────────────────┼───────────────────────────────────────────────────┤
  │ Choose          │ - Graph is large (100K+ nodes) — need sampling   │
  │ GraphSAGE when  │ - New nodes arrive at inference time (INDUCTIVE) │
  │                 │ - Mini-batch training required (GPU memory bound) │
  │                 │ - Example: social networks, recommendations,     │
  │                 │   dynamic knowledge graphs                       │
  └─────────────────┴───────────────────────────────────────────────────┘

  ACROSS ALL OF MODULE 5 — Architecture Selection Guide:

  ┌──────────────────┬──────────────────────────────────────────────────┐
  │ Data Type        │ Architecture                                     │
  ├──────────────────┼──────────────────────────────────────────────────┤
  │ Images           │ CNN / ViT + transfer learning (ImageNet)         │
  │ Text             │ Transformer + transfer learning (BERT / GPT)     │
  │ Sequences        │ LSTM / Transformer                               │
  │ Graphs           │ GNN (GCN / GAT / GraphSAGE — task dependent)    │
  │ Tabular          │ Gradient boosting (fast, reliable, no pretrain)  │
  └──────────────────┴──────────────────────────────────────────────────┘


In [ ]:
print("=" * 70)
print("  DECISION FRAMEWORK: Choosing a GNN Architecture")
print("=" * 70)
print(
)



## REGISTER — Best Model in ModelRegistry


In [ ]:
print("=" * 70)
print("  MODEL REGISTRATION")
print("=" * 70)

if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_gnn_{best_name.lower()}",
        model=best_model_obj,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val_acc),
            MetricSpec(name="best_test_accuracy", value=best_test_acc),
            MetricSpec(name="final_loss", value=results[best_name]["losses"][-1]),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {best_name}: version={version.version}")
    print(f"    val_acc={best_val_acc:.4f}, test_acc={best_test_acc:.4f}")
else:
    print("  ModelRegistry not available — skipping registration")



## FINAL SUMMARY


In [ ]:
print("\n" + "=" * 70)
print("  FINAL SUMMARY")
print("=" * 70)
print(
    f"\nDataset: {dataset_name} ({N} nodes, {graph_data['n_edges_undirected']} edges, "
    f"{n_classes} classes)"
)
print(f"\nNode Classification (best validation accuracy):")
for name, r in results.items():
    n_params = sum(p.numel() for p in r["model"].parameters())
    print(
        f"  {name:>12}: val={max(r['val_accs']):.4f}  "
        f"test={max(r['test_accs']):.4f}  params={n_params:,}"
    )
print(f"\nBest model: {best_name}")



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson walked the journey of graph neural networks — GCN, GAT,
# GraphSAGE — each with custom message-passing and aggregation logic.
# The kailash-ml SDK ships a single-call diagnostic primitive that
# closes the production loop: km.diagnose inspects a trained model and
# emits an auto-dashboard (loss curves, gradient flow, dead neurons,
# activation stats, weight distributions). One cell. Every diagnostic
# students would otherwise hand-roll, ready to surface in a Plotly
# dashboard.

from kailash_ml import diagnose

# GNN forward signatures take (X, A_norm) tuples. We feed an iterable of
# such tuples reusing the lesson's full-graph tensors. `kind='auto'`
# dispatches by model type — DLDiagnostics for torch.nn.Module.
graph_iter = [(X, A_norm) for _ in range(2)]
report = diagnose(best_model_obj, kind="auto", data=graph_iter, show=False)
report.plot_training_dashboard()
print()
print("km.diagnose: 1 line of code -> the same observability the lesson")
print("body hand-rolled in 200+ lines. This is what 'destination-first'")
print("means — when the journey is internalised, the SDK is one call.")



## REFLECTION


GNN ARCHITECTURES:
  [x] GCN: message passing as matrix multiplication  H' = A_norm @ H @ W
      Fixed aggregation via degree-normalised adjacency. Simplest and
      fastest. Works well on homogeneous graphs like citation networks.
  [x] GAT: learned attention weights over neighbours
      Each node decides how much to attend to each neighbour based on
      feature content. More expressive but more parameters. Interpretable.
  [x] GraphSAGE: sample + aggregate with separate self/neighbour projections
      INDUCTIVE — can generalise to unseen nodes at inference time.
      Neighbourhood sampling provides regularisation and scalability.

  GNN TASKS:
  [x] Node classification: predict the label of each node from its
      features and neighbourhood structure ({n_classes} classes on {dataset_name})
  [x] Link prediction: predict missing edges using dot-product decoder
      on GNN embeddings (foundation for recommendation systems)
  [x] Embedding visualisation: 2-D PCA projection shows class separation
      in the learned representation space

  ML ENGINEERING:
  [x] Tracked every GNN variant with ExperimentTracker (params, per-epoch
      loss, validation accuracy, test accuracy across {EPOCHS} epochs)
  [x] Registered best model in ModelRegistry with versioned metrics
  [x] Quantitative comparison table: parameters, accuracy, convergence
      speed — not eyeballing, but systematic tracked experiments

  SINGAPORE APPLICATIONS:
  [x] NUS/NTU research classification (GCN — citation graph)
  [x] PayNow/NETS fraud detection (GAT — interpretable attention)
  [x] GrabFood/foodpanda recommendations (GraphSAGE — scalable, inductive)
  [x] SGH drug-disease interaction discovery (link prediction)

  KEY INSIGHT: All three GNNs learn by aggregating information from
  neighbours, but they differ in HOW they aggregate:
    GCN        -> fixed weights (degree normalisation)
    GAT        -> learned weights (attention) + interpretability
    GraphSAGE  -> sampled + learned (separate self/neighbour) + inductive

  Next: In Exercise 7, you'll apply transfer learning with a pre-trained
  ResNet-18 to a new image classification task and export to ONNX...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Exercise 6 Complete")
print("=" * 70)
print(
)

# Clean up
await conn.close()



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Cross-architecture loss comparison
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F

    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (GNN Architecture Comparison (GCN vs GAT vs SAGE)) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        best_model,
        [(features, labels)],
        _diag_loss,
        title="GNN Architecture Comparison (GCN vs GAT vs SAGE)",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow: all 3 architectures healthy (RMS ~1e-3).
# [!] Over-smoothing detected at depth 4+: GCN worst (cosine sim 0.94),
#     GAT intermediate (0.87), SAGE best (0.79).



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [X-RAY — GNN-SPECIFIC] Over-smoothing is THE GNN scalability
#     problem. Cosine similarity ~1.0 across node embeddings means
#     the model can't distinguish nodes. Slide 5.6 addresses this.
#     >> Ranked by over-smoothing resistance:
#        SAGE (inductive aggregation) > GAT (learned attention) > GCN (fixed)
#     >> Prescription: depth 2-3 for GCN, up to 4 for GAT, up to 6+ for
#        SAGE with skip connections.
#
#  [STETHOSCOPE] All three converge to similar validation accuracy
#     on Cora — architecture choice matters more for SCALABILITY and
#     INDUCTIVE capability than raw accuracy.

